# Traffic Density Prediction Model

Notebook ini digunakan untuk melatih model Machine Learning yang dapat memprediksi tingkat kepadatan lalu lintas berdasarkan data historis 7 hari.
Model yang digunakan adalah **Random Forest Classifier**.

In [1]:
import pandas as pd
import glob
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Opsional: Untuk mengabaikan warning
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
Membaca seluruh file CSV di folder `collected_data` dan menggabungkannya.

In [2]:
# Tentukan direktori tempat data tersimpan
data_path = 'collected_data/*.csv'
file_list = glob.glob(data_path)

print(f"Menemukan {len(file_list)} file CSV.")

# Gabungkan semua CSV ke dalam satu DataFrame
dfs = []
for f in file_list:
    # Membaca CSV tanpa header
    df_temp = pd.read_csv(f, header=None)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

# Berikan nama kolom agar mudah dipahami
df.columns = ['id', 'timestamp', 'density_category', 'total_vehicles', 
              'c1', 'c2', 'c3', 'c4', 'c5']

print("Total baris data:", len(df))
df.head()

Menemukan 7 file CSV.
Total baris data: 84689


,id,timestamp,density_category,total_vehicles,c1,c2,c3,c4,c5
0,1,2026-05-29 06:06:37.739051,Empty,3,0,2,1,0,0
1,2,2026-05-29 06:06:43.303333,Empty,4,0,4,0,0,0
2,3,2026-05-29 06:06:49.162714,Empty,4,1,1,2,0,0
3,4,2026-05-29 06:06:55.120833,Empty,4,1,0,2,0,1
4,5,2026-05-29 06:07:00.466533,Empty,6,4,1,1,0,0


## 2. Feature Engineering
Mengekstrak informasi `Hari`, `Jam`, dan `Menit` dari kolom `timestamp` karena hanya waktu yang kita ketahui di masa depan untuk memprediksi.

In [3]:
# Konversi kolom timestamp menjadi tipe datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Ekstrak fitur
df['day_of_week'] = df['timestamp'].dt.dayofweek # Senin=0, Minggu=6
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute

# Kita hanya membutuhkan fitur waktu untuk prediksi masa depan
X = df[['day_of_week', 'hour', 'minute']]

# Target variabel yang ingin diprediksi
y = df['density_category']

print("Beberapa baris data input (X):")
print(X.head())
print("\nDistribusi Target (y):")
print(y.value_counts())

Beberapa baris data input (X):
   day_of_week  hour  minute
0            4     6       6
1            4     6       6
2            4     6       6
3            4     6       6
4            4     6       7

Distribusi Target (y):
density_category
Empty     54538
Low       28790
Medium     1361
Name: count, dtype: int64


## 3. Data Splitting (Train/Test Split)
Memisahkan sebagian data untuk pelatihan (80%) dan sebagian untuk pengujian (20%).

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data latih (Training) : {len(X_train)} baris")
print(f"Data uji (Testing)   : {len(X_test)} baris")

Data latih (Training) : 67751 baris
Data uji (Testing)   : 16938 baris


## 4. Model Training
Melatih model **Random Forest** menggunakan data latih.

In [5]:
# Inisiasi Model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Melatih Model
print("Sedang melatih model...")
model.fit(X_train, y_train)
print("Pelatihan selesai!")

Sedang melatih model...
Pelatihan selesai!


## 5. Model Evaluation
Menguji performa model terhadap data uji (Testing).

In [6]:
# Melakukan prediksi pada data pengujian
y_pred = model.predict(X_test)

# Menghitung akurasi
acc = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {acc * 100:.2f}%")

# Menampilkan laporan detail per kelas
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred))

Akurasi Model: 80.02%

Laporan Klasifikasi:
              precision    recall  f1-score   support

       Empty       0.84      0.88      0.86     10865
         Low       0.73      0.67      0.70      5820
      Medium       0.44      0.19      0.27       253

    accuracy                           0.80     16938
   macro avg       0.67      0.58      0.61     16938
weighted avg       0.79      0.80      0.80     16938



## 6. Export Model
Menyimpan model yang sudah dilatih menjadi file `.pkl` agar bisa digunakan di web server (Backend).

In [7]:
# Nama file tempat menyimpan model
model_filename = 'traffic_model.pkl'

# Menyimpan menggunakan joblib
joblib.dump(model, model_filename)

print(f"Model berhasil disimpan sebagai '{model_filename}'")

Model berhasil disimpan sebagai 'traffic_model.pkl'


## Selesai
Sekarang file `traffic_model.pkl` sudah siap. Di backend (misalnya API Flask/FastAPI), Anda bisa memuatnya dengan:
```python
import joblib
import pandas as pd

model = joblib.load('traffic_model.pkl')
# Untuk memprediksi:
# prediksi = model.predict(pd.DataFrame({'day_of_week': [0], 'hour': [8], 'minute': [30]}))
```